## Section 1: Import Required Libraries

We'll use NumPy for numerical operations, Matplotlib for visualization, Shapely for geometric operations, and the math module for trigonometric functions.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Circle
from shapely.geometry import Point, Polygon
import math

In [2]:
# Set default values for testing (you can replace with input() for interactive mode)

# Option 1: Use fixed values
H = 100.0  # Depth of cover (m)
h = 2.5    # Panel thickness (m)
e = 0.9    # Extraction ratio
a = 0.5    # Subsidence factor
NEW = 1.2  # NEW ratio
alpha_deg = 30.0  # Angle of draw (degrees)
grid_spacing = 5.0  # Mesh grid spacing (m)

# Option 2: Uncomment below for interactive input mode
# H = float(input("Enter Depth of Cover (H) in meters: "))
# h = float(input("Enter Panel Thickness (h) in meters: "))
# e = float(input("Enter Extraction Ratio (e.g., 0.7 for B&P, 1.0 for Longwall): "))
# a = float(input("Enter Subsidence Factor (a): "))
# NEW = float(input("Enter NEW ratio (W_new / H): "))
# alpha_deg = float(input("Enter Angle of draw (alpha) in degrees: "))
# grid_spacing = float(input("Enter mesh grid spacing (e.g., 5 or 10 meters): "))

# Convert angle to radians
alpha_rad = math.radians(alpha_deg)

# Calculate maximum subsidence
S_max = h * e * a

print(f"\n{'='*60}")
print(f"Mining Subsidence Analysis Parameters")
print(f"{'='*60}")
print(f"Depth of Cover (H):        {H:.2f} m")
print(f"Panel Thickness (h):       {h:.2f} m")
print(f"Extraction Ratio (e):      {e:.2f}")
print(f"Subsidence Factor (a):     {a:.2f}")
print(f"NEW Ratio:                 {NEW:.2f}")
print(f"Angle of Draw (α):         {alpha_deg:.2f}° ({alpha_rad:.4f} rad)")
print(f"Grid Spacing:              {grid_spacing:.2f} m")
print(f"Maximum Subsidence (S_max):{S_max:.4f} m")
print(f"{'='*60}\n")


Mining Subsidence Analysis Parameters
Depth of Cover (H):        100.00 m
Panel Thickness (h):       2.50 m
Extraction Ratio (e):      0.90
Subsidence Factor (a):     0.50
NEW Ratio:                 1.20
Angle of Draw (α):         30.00° (0.5236 rad)
Grid Spacing:              5.00 m
Maximum Subsidence (S_max):1.1250 m



In [3]:
# Define panel coordinates (easting, northing) - example rectangular panel
# You can modify these coordinates or input from external data
panel_coords = [
    (200, 100),   # Southwest corner
    (500, 100),   # Southeast corner
    (500, 400),   # Northeast corner
    (200, 400),   # Northwest corner
]

# Create Shapely Polygon
panel_poly = Polygon(panel_coords)

# Get bounding box
min_x, min_y, max_x, max_y = panel_poly.bounds

print(f"\nPanel Geometry:")
print(f"  Panel Bounds: X=[{min_x:.1f}, {max_x:.1f}], Y=[{min_y:.1f}, {max_y:.1f}]")
print(f"  Panel Area: {panel_poly.area:.0f} m²")

# Calculate influence radius
R = H * math.tan(alpha_rad)
print(f"  Influence Radius (R): {R:.2f} m")

# Expand grid bounds by R (the influence radius)
grid_min_x = min_x - R
grid_max_x = max_x + R
grid_min_y = min_y - R
grid_max_y = max_y + R

print(f"\nGrid Bounds (expanded by R):")
print(f"  X: [{grid_min_x:.1f}, {grid_max_x:.1f}]")
print(f"  Y: [{grid_min_y:.1f}, {grid_max_y:.1f}]")

# Create 1D arrays for X and Y
x_array = np.arange(grid_min_x, grid_max_x + grid_spacing, grid_spacing)
y_array = np.arange(grid_min_y, grid_max_y + grid_spacing, grid_spacing)

# Create 2D meshgrid
X_grid, Y_grid = np.meshgrid(x_array, y_array)

print(f"  Grid Points: {len(x_array)} × {len(y_array)} = {X_grid.size} points")
print()


Panel Geometry:
  Panel Bounds: X=[200.0, 500.0], Y=[100.0, 400.0]
  Panel Area: 90000 m²
  Influence Radius (R): 57.74 m

Grid Bounds (expanded by R):
  X: [142.3, 557.7]
  Y: [42.3, 457.7]
  Grid Points: 85 × 85 = 7225 points



In [4]:
# --- CORRECTED SECTION 4 ---

# Pre-calculated Ring Weighting Factors (S_i) from the center outwards
# These integrate the bell-shaped influence curve over the 10 rings
S_i = [0.0297, 0.0854, 0.1311, 0.1607, 0.1708, 0.1596, 0.1294, 0.0859, 0.0397, 0.0069]

num_rings = 10
num_sectors = 64
template_elements = []

# For each ring
for i in range(num_rings):
    # Calculate mid-radius for this ring
    r_mid = (i + 0.5) * (R / num_rings)

    # Calculate the element weight for this specific ring
    weight = S_i[i] / num_sectors

    # For each sector
    for j in range(num_sectors):
        # Calculate angle
        phi = j * (2 * np.pi / num_sectors)

        # Local coordinates relative to origin
        dx = r_mid * np.cos(phi)
        dy = r_mid * np.sin(phi)

        template_elements.append((dx, dy, weight))

print(f"Influence Template Built:")
print(f"  Total Elements: {len(template_elements)}")
print(f"  Rings: {num_rings}, Sectors: {num_sectors}")
print(f"  Sum of all weights (Should be 1.0): {sum([w for x, y, w in template_elements]):.4f}")
print(f"  Max Radius: {R:.2f} m\n")

Influence Template Built:
  Total Elements: 640
  Rings: 10, Sectors: 64
  Sum of all weights (Should be 1.0): 0.9992
  Max Radius: 57.74 m



In [5]:
import time

# Main calculation: loop through all grid points
results = []
total_points = X_grid.size

print("Starting Subsidence Calculation...")
print(f"Evaluating {total_points} grid points...\n")

start_time = time.time()

# Flatten the grid arrays
X_flat = X_grid.flatten()
Y_flat = Y_grid.flatten()

# Process each grid point
for idx, (X_p, Y_p) in enumerate(zip(X_flat, Y_flat)):
    point_subsidence = 0.0
    
    # Check all template elements
    for dx, dy, weight in template_elements:
        # Translate template element to current grid point
        element_x = X_p + dx
        element_y = Y_p + dy
        
        # Create point at this location
        pt = Point(element_x, element_y)
        
        # Check if point is inside the panel
        if panel_poly.contains(pt):
            # This element contributes to subsidence
            point_subsidence += weight * S_max
    
    # Store result
    results.append((X_p, Y_p, point_subsidence))
    
    # Progress indicator
    if (idx + 1) % max(1, total_points // 10) == 0:
        elapsed = time.time() - start_time
        percent = 100 * (idx + 1) / total_points
        est_total = elapsed / ((idx + 1) / total_points)
        est_remaining = est_total - elapsed
        print(f"  {percent:5.1f}% ({idx+1:6d}/{total_points}) - "
              f"Elapsed: {elapsed:6.1f}s, Remaining: ~{est_remaining:6.1f}s")

elapsed_time = time.time() - start_time
print(f"\nCalculation Complete!")
print(f"  Total Time: {elapsed_time:.2f} seconds")
print(f"  Points/sec: {total_points/elapsed_time:.0f}\n")

Starting Subsidence Calculation...
Evaluating 7225 grid points...

   10.0% (   722/7225) - Elapsed:    5.7s, Remaining: ~  51.5s
   20.0% (  1444/7225) - Elapsed:   11.8s, Remaining: ~  47.1s
   30.0% (  2166/7225) - Elapsed:   18.2s, Remaining: ~  42.4s
   40.0% (  2888/7225) - Elapsed:   24.7s, Remaining: ~  37.1s
   50.0% (  3610/7225) - Elapsed:   31.1s, Remaining: ~  31.1s
   60.0% (  4332/7225) - Elapsed:   37.4s, Remaining: ~  25.0s
   70.0% (  5054/7225) - Elapsed:   43.8s, Remaining: ~  18.8s
   79.9% (  5776/7225) - Elapsed:   50.1s, Remaining: ~  12.6s
   89.9% (  6498/7225) - Elapsed:   56.0s, Remaining: ~   6.3s
   99.9% (  7220/7225) - Elapsed:   61.7s, Remaining: ~   0.0s

Calculation Complete!
  Total Time: 61.70 seconds
  Points/sec: 117



## Section 6: Extract and Prepare Results Data

Convert 1D results list into 2D grids compatible with Matplotlib's contouring functions.

In [6]:
import pandas as pd

# Convert results to a DataFrame for easy viewing and exporting
df_results = pd.DataFrame(results, columns=['Easting_X', 'Northing_Y', 'Subsidence_S'])

# Filter out points with zero subsidence to keep the data clean (optional)
df_impacted = df_results[df_results['Subsidence_S'] > 0.001].copy()

print("Sample of Calculated Points (Easting, Northing, Subsidence):")
print(df_impacted.head(15).to_string(index=False))
print(f"\nTotal points with subsidence: {len(df_impacted)} / {len(df_results)}")

# Save to CSV so you can view it in Excel or use it in other software
df_results.to_csv('subsidence_results.csv', index=False)
print("\nAll point values have been saved to 'subsidence_results.csv'")

Sample of Calculated Points (Easting, Northing, Subsidence):
 Easting_X  Northing_Y  Subsidence_S
192.264973   52.264973      0.001183
197.264973   52.264973      0.002002
202.264973   52.264973      0.002821
207.264973   52.264973      0.003640
212.264973   52.264973      0.004460
217.264973   52.264973      0.004581
222.264973   52.264973      0.004702
227.264973   52.264973      0.004823
232.264973   52.264973      0.004823
237.264973   52.264973      0.004823
242.264973   52.264973      0.004823
247.264973   52.264973      0.004823
252.264973   52.264973      0.004823
257.264973   52.264973      0.004823
262.264973   52.264973      0.004823

Total points with subsidence: 6296 / 7225

All point values have been saved to 'subsidence_results.csv'


In [7]:
# --- REVISED SECTION 7: INTERACTIVE VISUALIZATION ---
import plotly.graph_objects as go
import numpy as np

fig = go.Figure()

# 1. Plot the Grid Points with Interactive Hover
fig.add_trace(go.Scatter(
    x=df_results['Easting_X'],
    y=df_results['Northing_Y'],
    mode='markers',
    marker=dict(
        size=7,
        color=df_results['Subsidence_S'], # Color points by subsidence value
        colorscale='Viridis',
        reversescale=True,
        showscale=True,
        colorbar=dict(title="Subsidence (m)")
    ),
    text=df_results['Subsidence_S'],
    hovertemplate=
    "<b>Easting:</b> %{x:.2f}<br>" +
    "<b>Northing:</b> %{y:.2f}<br>" +
    "<b>Subsidence:</b> %{text:.4f} m<extra></extra>", # <extra></extra> hides secondary trace labels
    name="Grid Points"
))

# 2. Plot the Panel Boundary
panel_x, panel_y = panel_poly.exterior.xy
fig.add_trace(go.Scatter(
    x=list(panel_x),
    y=list(panel_y),
    mode='lines',
    line=dict(color='black', width=4),
    hoverinfo='skip',
    name='Panel Boundary'
))

# 3. Plot the Grid Boundary (Expanded by R)
fig.add_trace(go.Scatter(
    x=[grid_min_x, grid_max_x, grid_max_x, grid_min_x, grid_min_x],
    y=[grid_min_y, grid_min_y, grid_max_y, grid_max_y, grid_min_y],
    mode='lines',
    line=dict(color='blue', width=2, dash='dash'),
    hoverinfo='skip',
    name=f'Grid Boundary (+{R:.1f}m)'
))

# 4. Draw Sample Influence Circle (10 Rings, 64 Sectors)
# Centering the sample circle in the middle of the panel boundary
center_x = (min_x + max_x) / 2
center_y = (min_y + max_y) / 2

# Draw the 10 Rings
for i in range(1, 11):
    radius = i * (R / 10)
    fig.add_shape(
        type="circle",
        x0=center_x - radius, y0=center_y - radius,
        x1=center_x + radius, y1=center_y + radius,
        line=dict(color="red", width=1, dash="dot" if i < 10 else "solid")
    )

# Draw the 64 Sectors (Drawing 32 lines completely across the diameter creates 64 slices)
for i in range(32):
    angle = i * (np.pi / 32)
    dx = R * np.cos(angle)
    dy = R * np.sin(angle)
    fig.add_shape(
        type="line",
        x0=center_x - dx, y0=center_y - dy,
        x1=center_x + dx, y1=center_y + dy,
        line=dict(color="red", width=0.5, dash="dot")
    )

# 5. Format the Layout
fig.update_layout(
    title="Interactive Subsidence Grid (Hover for Data)",
    xaxis_title="Easting (X)",
    yaxis_title="Northing (Y)",
    yaxis=dict(scaleanchor="x", scaleratio=1), # Forces Equal Aspect Ratio (Perfect Circles)
    width=900,
    height=800,
    plot_bgcolor='white',
    hovermode="closest",
    legend=dict(yanchor="top", y=0.99, xanchor="left", x=0.01) # Move legend to top left
)

# Add faint gridlines
fig.update_xaxes(showgrid=True, gridwidth=1, gridcolor='LightGray')
fig.update_yaxes(showgrid=True, gridwidth=1, gridcolor='LightGray')

# Display the interactive plot
fig.show()